# 2장 — 곱의 세계: 로그 눈금과 멱법칙 🧾

웹 랩의 "종이 갈아끼우기"를 코드로 재현합니다. 세 곡선을 세 종이에 전부 그려서
**어떤 곡선이 어느 종이에서 직선이 되는지** 한 장에 확인하고, 직선의 **기울기에서 법칙을
역으로 읽는** 연습(3장 예습)까지 해 봅니다.

- 일차 `y = 10x`: 같은 간격만큼 **더해짐**
- 거듭제곱(멱법칙) `y = x^1.5`: x 10배마다 y는 10^1.5 배
- 지수 `y = 2^(x/10)`: x가 +10 될 때마다 **×2**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch2')

## 1. 종이 갈아끼우기: 3곡선 × 3종이 한판 대조

행 = 곡선, 열 = 종이. **각 행에서 직선으로 펴지는 칸**(초록)이 그 곡선의 "맞는 종이"입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

x = np.logspace(0, 2, 50)  # 1 → 100 (로그 간격 샘플)
curves = [
    ('일차  y = 10x', 10 * x, 'linlin'),
    ('거듭제곱  y = x^1.5', x ** 1.5, 'loglog'),
    ('지수  y = 2^(x/10)', 2 ** (x / 10), 'semilog'),
]
papers = [('보통 종이', 'linlin'), ('반로그 (y만 로그)', 'semilog'), ('로그-로그', 'loglog')]

fig, axes = plt.subplots(3, 3, figsize=(11, 8.5))
for r, (cname, y, straight_on) in enumerate(curves):
    for c, (pname, pid) in enumerate(papers):
        ax = axes[r][c]
        ax.plot(x, y, lw=2, color='#059669' if pid == straight_on else '#6366f1')
        if pid in ('semilog', 'loglog'):
            ax.set_yscale('log')
        if pid == 'loglog':
            ax.set_xscale('log')
        if pid == straight_on:
            ax.set_facecolor('#ecfdf5')
            ax.set_title(f'{pname} → 📏 직선!', fontsize=10, color='#047857')
        else:
            ax.set_title(pname, fontsize=10, color='#94a3b8')
        if c == 0:
            ax.set_ylabel(cname, fontsize=10)
        ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()
print('일차↔보통, 지수↔반로그, 멱법칙↔로그-로그. 종이가 곡선의 성질을 드러낸다')

## 2. 직선에서 법칙을 "역으로 읽기": 기울기 실측

로그를 씌우면 `y = a·xᵇ` 는 `log y = log a + b·log x`, 곧 로그-로그 종이 위의 직선입니다.
그렇다면 반대로, **직선에 자를 대면 b(기울기)와 a(절편)를 읽어낼 수 있다**는 뜻:
`np.polyfit` 으로 실제로 재 봅니다. 3장에서 손실 곡선에 그대로 쓸 기술이에요.

In [ ]:
# 멱법칙 곡선의 로그-로그 기울기 — 정확히 1.5가 나와야 한다
b_fit, loga_fit = np.polyfit(np.log10(x), np.log10(x ** 1.5), 1)
print(f'멱법칙 y = x^1.5  → 로그-로그 기울기 b = {b_fit:.6f}, 절편 log a = {loga_fit:.6f}')
assert abs(b_fit - 1.5) < 1e-9 and abs(loga_fit) < 1e-9, '기울기 = 지수 b, 절편 = log a(=log 1=0)'

# 지수 곡선은 "반로그"에서 직선 — 기울기는 log10(2)/10 (x가 +10마다 ×2)
s_fit, _ = np.polyfit(x, np.log10(2 ** (x / 10)), 1)
assert abs(s_fit - np.log10(2) / 10) < 1e-9
print(f'지수 y = 2^(x/10) → 반로그 기울기 = {s_fit:.6f} (= log10(2)/10)')

# "x 10배 → y는 10^b 배" 감각 표 — 웹 랩 기울기 다이얼과 동일
print()
print('기울기 b | x 10배 → y는?')
print('-' * 30)
for b in [1.5, 1.0, 0.5, 0.3, -0.1]:
    f = 10 ** b
    note = f'{f:.2f}배' if f >= 1 else f'{f:.2f}배 (≈{(1-f)*100:.0f}% 감소)'
    print(f'{b:>7} | {note}')

assert round(10 ** 0.5, 2) == 3.16
assert round(10 ** 0.3, 2) == 2.0, '10^0.3 ≈ 2 — "10배당 2배" 어림'
assert round(10 ** -0.1, 2) == 0.79, '10배 부어도 21% 감소'
print()
print('✔ 본문 수치 검증: b=0.5→3.16배, b=0.3→약 2배, b=−0.1→0.79배')

## 정리: 오늘 가져갈 세 문장

1. 곱의 세계(10배씩 뛰는 수)는 **로그 눈금**으로 그린다. 같은 간격 = 같은 배수.
2. **멱법칙 y = a·xᵇ 는 로그-로그 종이에서 직선**: 기울기가 b, 절편이 log a.
3. 기울기 읽는 법: **x 10배 → y는 10ᵇ배** (b=0.3이면 약 2배, b=−0.1이면 21% 감소).

다음 **3장(손실의 직선)**: 미니 LM에 데이터를 10배씩 늘려 먹이며 손실을 실측하고,
로그-로그 종이 위에서 `np.polyfit` 으로 **스케일링 법칙의 기울기를 직접 잽니다**.